# Gini Impurity: Calculate Gini for All Splits

**Dataset:** Play Tennis (14 records)  
**Target column:** `PlayTennis` (`Yes` / `No`)

This notebook evaluates **every unique binary split** for each input feature. For a feature with categories A, B, C, it checks A vs (B, C), B vs (A, C), and C vs (A, B), without counting reversed duplicates.

For each split, it calculates:
1. Parent Gini impurity
2. Left child Gini impurity
3. Right child Gini impurity
4. Weighted child Gini impurity
5. Gini reduction (parent Gini − weighted child Gini)

The notebook uses only pandas and Python's standard library.

In [1]:
import pandas as pd
from itertools import combinations

# Load the dataset from the same folder as this notebook.
DATASET_FILE = "play_tennis_dataset.csv"
df = pd.read_csv(DATASET_FILE)

TARGET = "PlayTennis"
FEATURES = [column for column in df.columns if column != TARGET]

print("Dataset shape:", df.shape)
display(df.head())
print("\\nTarget class counts:")
print(df[TARGET].value_counts())

Dataset shape: (14, 5)


,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


\nTarget class counts:
PlayTennis
Yes    9
No     5
Name: count, dtype: int64


## 1. Gini impurity formula

For a node containing classes with proportions \(p_1, p_2, \\ldots, p_k\):

\\[
Gini = 1 - \\sum_{i=1}^{k} p_i^2
\\]

For a binary split:

\\[
Weighted\\ Child\\ Gini =
\\frac{N_L}{N}Gini_L + \\frac{N_R}{N}Gini_R
\\]

\\[
Gini\\ Reduction = Gini_{Parent} - Weighted\\ Child\\ Gini
\\]

A larger Gini reduction means the split reduces impurity more.

In [2]:
def gini_impurity(labels):
    """Calculate Gini impurity for a collection of class labels."""
    if len(labels) == 0:
        return 0.0
    probabilities = labels.value_counts(normalize=True)
    return float(1 - (probabilities ** 2).sum())


parent_gini = gini_impurity(df[TARGET])
print(f"Number of records: {len(df)}")
print("Parent class counts:")
print(df[TARGET].value_counts())
print(f"\\nParent Gini impurity: {parent_gini:.4f}")

Number of records: 14
Parent class counts:
PlayTennis
Yes    9
No     5
Name: count, dtype: int64
\nParent Gini impurity: 0.4592


## 2. Generate and calculate every unique binary split

For each feature, all non-empty category groupings are considered. To avoid duplicate reversed splits (for example, A vs B and B vs A), only one representative of each partition is evaluated.

In [3]:
def unique_binary_splits(categories):
    """Return each unique non-empty binary partition exactly once."""
    categories = sorted(list(categories))
    n = len(categories)

    # Fix the first category on the left to avoid reversed duplicates.
    # Exclude the full set, which would leave an empty right child.
    first = categories[0]
    remaining = categories[1:]

    splits = []
    for size in range(0, len(remaining) + 1):
        for subset in combinations(remaining, size):
            left_categories = {first, *subset}
            right_categories = set(categories) - left_categories
            if right_categories:
                splits.append((left_categories, right_categories))
    return splits


results = []

for feature in FEATURES:
    categories = df[feature].dropna().unique()

    for left_categories, right_categories in unique_binary_splits(categories):
        left_mask = df[feature].isin(left_categories)
        left_df = df[left_mask]
        right_df = df[~left_mask]

        left_gini = gini_impurity(left_df[TARGET])
        right_gini = gini_impurity(right_df[TARGET])

        n_left = len(left_df)
        n_right = len(right_df)
        total = n_left + n_right

        weighted_child_gini = (
            (n_left / total) * left_gini
            + (n_right / total) * right_gini
        )
        gini_reduction = parent_gini - weighted_child_gini

        results.append({
            "Feature": feature,
            "Left split values": ", ".join(sorted(map(str, left_categories))),
            "Right split values": ", ".join(sorted(map(str, right_categories))),
            "Left records": n_left,
            "Right records": n_right,
            "Parent Gini": parent_gini,
            "Left child Gini": left_gini,
            "Right child Gini": right_gini,
            "Weighted child Gini": weighted_child_gini,
            "Gini reduction": gini_reduction
        })

results_df = pd.DataFrame(results)

# Display every split with values rounded to four decimal places.
numeric_columns = [
    "Parent Gini", "Left child Gini", "Right child Gini",
    "Weighted child Gini", "Gini reduction"
]
display(results_df.style.format({column: "{:.4f}" for column in numeric_columns}))

,Feature,Left split values,Right split values,Left records,Right records,Parent Gini,Left child Gini,Right child Gini,Weighted child Gini,Gini reduction
0,Outlook,Overcast,"Rain, Sunny",4,10,0.4592,0.0000,0.5000,0.3571,0.1020
1,Outlook,"Overcast, Rain",Sunny,9,5,0.4592,0.3457,0.4800,0.3937,0.0655
2,Outlook,"Overcast, Sunny",Rain,9,5,0.4592,0.4444,0.4800,0.4571,0.0020
3,Temperature,Cool,"Hot, Mild",4,10,0.4592,0.3750,0.4800,0.4500,0.0092
4,Temperature,"Cool, Hot",Mild,8,6,0.4592,0.4688,0.4444,0.4583,0.0009
5,Temperature,"Cool, Mild",Hot,10,4,0.4592,0.4200,0.5000,0.4429,0.0163
6,Humidity,High,Normal,7,7,0.4592,0.4898,0.2449,0.3673,0.0918
7,Wind,Strong,Weak,6,8,0.4592,0.5000,0.3750,0.4286,0.0306


## 3. Best split

The best split is the one with the **highest Gini reduction** (equivalently, the lowest weighted child Gini impurity).

In [4]:
best_split = results_df.loc[results_df["Gini reduction"].idxmax()]

print("Best split based on Gini reduction")
print("----------------------------------")
for column in results_df.columns:
    value = best_split[column]
    if isinstance(value, float):
        print(f"{column}: {value:.4f}")
    else:
        print(f"{column}: {value}")

Best split based on Gini reduction
----------------------------------
Feature: Outlook
Left split values: Overcast
Right split values: Rain, Sunny
Left records: 4
Right records: 10
Parent Gini: 0.4592
Left child Gini: 0.0000
Right child Gini: 0.5000
Weighted child Gini: 0.3571
Gini reduction: 0.1020


## 4. Save the complete results

This creates a second CSV containing all calculated split values in the notebook's working folder.

In [5]:
results_df.to_csv("gini_all_splits_results.csv", index=False)
print("Saved: gini_all_splits_results.csv")
print("Total unique binary splits evaluated:", len(results_df))

Saved: gini_all_splits_results.csv
Total unique binary splits evaluated: 8


## 5. Conclusion

- **Parent Gini** measures impurity before splitting.
- **Left child Gini** and **right child Gini** measure impurity in each resulting group.
- **Weighted child Gini** combines both child impurities according to their sizes.
- **Gini reduction** measures how much impurity is removed by the split.
- The split with the highest Gini reduction is preferred.

**Note:** The dataset file `play_tennis_dataset.csv` must be in the same folder as this notebook before running it.